In [1]:
import seaborn as sns
import pandas as pd

# 1. Chargement et conversion des colonnes en datetime
taxis = sns.load_dataset("taxis")
print(f"Dimensions initiales : {taxis.shape}")

taxis["pickup"] = pd.to_datetime(taxis["pickup"])
taxis["dropoff"] = pd.to_datetime(taxis["dropoff"])

print(f"Période couverte : du {taxis['pickup'].min()} au {taxis['pickup'].max()}")

# 2. Nombre de courses et chiffre d'affaires par jour
activite_par_jour = (
    taxis.set_index("pickup")
    .resample("D")
    .agg(nb_courses=("total", "count"), chiffre_affaires=("total", "sum"))
    .round(2)
)
print("\n--- 2. Activité quotidienne (5 premiers jours) ---")
print(activite_par_jour.head())

# 3. Heure de la journée la plus chargée
taxis["heure"] = taxis["pickup"].dt.hour
courses_par_heure = taxis["heure"].value_counts().sort_index()
heure_plus_chargee = taxis["heure"].value_counts().idxmax()
print(f"\n--- 3. Heure la plus chargée ---")
print(f"Heure de pointe : {heure_plus_chargee}h (avec {taxis['heure'].value_counts().max()} courses)")

# 4. Jour de la semaine avec le pourboire moyen le plus élevé
taxis["jour_semaine"] = taxis["pickup"].dt.day_name()
pourboire_par_jour = taxis.groupby("jour_semaine")["tip"].mean().round(2)
jour_max_pourboire = pourboire_par_jour.idxmax()
print("\n--- 4. Pourboire moyen par jour ($) ---")
print(pourboire_par_jour)
print(f"\nJour le plus généreux : {jour_max_pourboire} ({pourboire_par_jour.max()} $)")

# 5. Calcul de la durée en minutes et traitement des anomalies
taxis["duree_min"] = (taxis["dropoff"] - taxis["pickup"]).dt.total_seconds() / 60
print("\n--- 5. Statistiques descriptives des durées (en minutes) ---")
print(taxis["duree_min"].describe().round(2))

nb_nulles = (taxis["duree_min"] <= 0).sum()
print(f"\nCourses aberrantes (durée <= 0 min) : {nb_nulles}")

# Traitement : suppression des courses nulles (annulations ou erreurs d'horodatage)
taxis_propre = taxis[taxis["duree_min"] > 0].copy()
print(f"Courses conservées après nettoyage : {len(taxis_propre)} (sur {len(taxis)})")


Dimensions initiales : (6433, 14)
Période couverte : du 2019-02-28 23:29:03 au 2019-03-31 23:43:45

--- 2. Activité quotidienne (5 premiers jours) ---
            nb_courses  chiffre_affaires
pickup                                  
2019-02-28           1              6.30
2019-03-01         241           4213.83
2019-03-02         198           3319.02
2019-03-03         169           3027.32
2019-03-04         171           3269.08

--- 3. Heure la plus chargée ---
Heure de pointe : 18h (avec 417 courses)

--- 4. Pourboire moyen par jour ($) ---
jour_semaine
Friday       1.98
Monday       2.03
Saturday     1.82
Sunday       1.91
Thursday     2.06
Tuesday      2.00
Wednesday    2.09
Name: tip, dtype: float64

Jour le plus généreux : Wednesday (2.09 $)

--- 5. Statistiques descriptives des durées (en minutes) ---
count    6433.00
mean       14.35
std        11.64
min         0.00
25%         6.50
50%        10.90
75%        18.52
max       107.67
Name: duree_min, dtype: float64

Course

### 5. Analyse et traitement des durées aberrantes

1. **Diagnostic** :
   - La durée médiane d'une course est de **10.9 minutes** (moyenne à 14.3 min).
   - **Anomalie détectée** : **6 courses** ont une durée de **0.0 minute** (`dropoff == pickup`), correspondant à des prises en charge immédiatement annulées ou des erreurs d'horodatage.
   - **Validation des valeurs hautes** : La durée maximale enregistrée est de **107.7 minutes** (~1h48). Cette valeur est cohérente pour un trajet métropolitain encombré vers un aéroport en heure de pointe ; elle est donc conservée.

2. **Décision de nettoyage** :
   - Les 6 courses de durée nulle sont exclues via le filtre `duree_min > 0`.
   - L'échantillon final exploitable comprend **6 427 courses**.
